# Installation PyPSA-Eur

This notebook walks through the full PyPSA-Eur workflow: setup, running the model, and analysing results from a solved sector-coupled network.

## Part 1 - Loading the network

Results are stored as NetCDF files under `pypsa-eur/results/networks/`.
The filename encodes the configuration: `base_s_***.nc`

In [1]:
import pypsa
import pandas as pd
import matplotlib.pyplot as plt

n = pypsa.Network("pypsa-eur/resources/networks/base_s_1_elec.nc")
print(n)

INFO:pypsa.network.io:Imported network 'Unnamed Network' has buses, carriers, generators, links, loads, storage_units, stores, sub_networks


PyPSA Network 'Unnamed Network'


# Conversion to GEMS format

## Part 2 - Imports & Logging

Firstly we import the Python package ; [pypsa-to-gems-converter](https://pypi.org/project/pypsa-to-gems-converter/)

In [1]:
pip install pypsa-to-gems-converter --quiet

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from src.pypsa_converter import PyPSAStudyConverter
import logging
from pathlib import Path
import pypsa
from pypsa import Network
import importlib.metadata


# Configure logging so progress is visible in the notebook
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(name)s – %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("pypsa_to_gems")

study_dir = Path("tmp/my_study")  # Absolute path to the GEMS study directory

PyPSA_file_path = "pypsa-eur/resources/networks/base_s_1_elec.nc"  # Absolute path to the PyPSA file
network = Network(PyPSA_file_path)
print(network)  # Print the network to verify it loaded correctly

10:43:45  WARNING   pypsa.network.io – Importing network from PyPSA version v1.1.2 while current version is v1.2.0. Read the release notes at `https://go.pypsa.org/release-notes` to prepare your network for import.
10:43:45  INFO      pypsa.network.io – New version 1.2.2 available! (Current: 1.2.0)
10:43:45  INFO      pypsa.network.io – Imported network 'Unnamed Network' has buses, carriers, generators, links, loads, storage_units, stores, sub_networks


PyPSA version : 1.2.0
pypsa-to-gems-converter version : 0.0.1
PyPSA Network 'Unnamed Network'


## Part 3 - Conversion PYPSA network to GEMS

In [ ]:
# Convert PyPSA network to GEMS
converter = PyPSAStudyConverter(
    pypsa_network=network,
    logger=logger,
    study_dir=study_dir,
    series_file_format=".tsv",  # Supported formats: .tsv, .csv
).to_gems_study()

## Part 4 - Inspect the GEMS Study

In [ ]:
import yaml
from collections import defaultdict

gems_study_file_path = Path(study_dir) / "systems" / "input"

system_yml_files = list(gems_study_file_path.rglob("system.yml"))
if not system_yml_files:
    print("system.yml not found.")
else:
    data = yaml.safe_load(system_yml_files[0].read_text())["system"]

    components = data.get("components", [])
    connections = data.get("connections", [])

    # Group components by model type
    by_model = defaultdict(list)
    for c in components:
        by_model[c["model"]].append(c["id"])

    print(f"System : {data['id']}")
    print(f"\nComponents ({len(components)} total)")
    print("-" * 20)
    for model, ids in sorted(by_model.items()):
        short = model.split(".")[-1]
        print(f"  [{short}]  ({len(ids)})")
        for cid in ids:
            print(f"    - {cid}")

    print(f"\nConnections ({len(connections)} total)")
    print("-" * 20)
    for conn in connections:
        print(f"  {conn.get('component1')}.{conn.get('port1')}  →  {conn.get('component2')}.{conn.get('port2')}")

## Part 5 - Run the GEMS optimization

In this section, we use antares modeler for running the GEMS converted simulation

In [ ]:
# Path to the Antares modeler binary
from asyncio import subprocess
from subprocess import run as sp_run 

import shutil
modeler_bin = Path(shutil.which("antares-modeler") or "/usr/local/bin/antares-modeler")
                                                            
sp_run([
      str(modeler_bin),
      str(study_dir / "systems")
  ])

## Part 6 - Analyse the simulation results

The solver writes a `simulation_table--<timestamp>.csv` file under `systems/output/`.  
Below we load the most recent one and visualise:
- the **objective value** (total system cost)
- the **installed capacity** (`p_nom`) per generator technology
- the **dispatch** (`p`) time series per technology

In [ ]:
import glob
import pandas as pd
import matplotlib.pyplot as plt

# ── Load the most recent simulation table ──────────────────────────────────────
output_dir = study_dir / "systems" / "output"
csv_files = sorted(glob.glob(str(output_dir / "simulation_table--*.csv")))
assert csv_files, f"No simulation_table CSV found in {output_dir}"
sim_csv = csv_files[-1]
print(f"Reading: {sim_csv}")

df = pd.read_csv(sim_csv)

# ── Objective value ────────────────────────────────────────────────────────────
obj_row = df[df["output"] == "OBJECTIVE_VALUE"]
if not obj_row.empty:
    obj_value = obj_row["value"].iloc[0]
    print(f"\nObjective value (total system cost): {obj_value:,.0f} €")

# ── Installed capacity (p_nom) per generator technology ───────────────────────
p_nom_df = df[
    (df["output"] == "p_nom") &
    (df["component"].str.startswith("generator_", na=False))
].copy()

# Extract technology name: last segment after the final "_"
p_nom_df["technology"] = p_nom_df["component"].str.split("_").str[-1]
capacity = p_nom_df.groupby("technology")["value"].sum().sort_values(ascending=False)
capacity = capacity[capacity > 0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(capacity.index, capacity.values / 1e3, color="steelblue")
axes[0].set_xlabel("Installed capacity (GW)")
axes[0].set_title("Generator installed capacity (p_nom)")
axes[0].invert_yaxis()

# ── Dispatch time series (p) per technology ───────────────────────────────────
p_df = df[
    (df["output"] == "p") &
    (df["component"].str.startswith("generator_", na=False)) &
    (df["absolute_time_index"].notna())
].copy()

p_df["technology"] = p_df["component"].str.split("_").str[-1]
p_df["t"] = pd.to_numeric(p_df["absolute_time_index"])

dispatch = (
    p_df.groupby(["t", "technology"])["value"]
    .sum()
    .unstack("technology")
    .fillna(0)
    .clip(lower=0)          # ignore small negative solver artefacts
)
dispatch = dispatch / 1e3   # MW → GW

dispatch.plot.area(ax=axes[1], linewidth=0)
axes[1].set_xlabel("Time step")
axes[1].set_ylabel("Generation (GW)")
axes[1].set_title("Dispatch by technology")
axes[1].legend(loc="upper right", fontsize=7, ncol=2)

plt.tight_layout()
plt.show()